# File 02/8

### DESCRIPTION:
- Filters tense forms based on morphological tags. Simple and compound verb forms can be identified and filtered.

#### NOTE: 
- There is a mismatch between the predefined morphology schema (all possible values) and the empirical data (only a subset is attested). 
- For **compound tenses**, this can be explained by the fact that auxiliary verb(s) and main verb are annotated separately.
- Thus the user has to find the main verb and its corresponding auxiliary verb(s), e.g.:

##### Verbs that can be found with a simple lookup like "--a-------" in column "Morphology"
```
AORIST
  Structure:
    Aux1: —
    Aux2: —
    Main: aorist
  Example:
    stavich''
```
##### Compound tense verbs that can be found by examining 
- a) the present form of auxiliary verb 'byti' ("--p-------") plus
- b) the resultative form of the main verb ("--s-------")

```
PERFECT
  Structure:
    Aux1: present (byti)
    Aux2: —
    Main: resultative
  Example:
    jesm' (po)stavil''
```


#### INPUT:
- Annotated tokens with morphological feature strings (fixed-position encoding)

#### OUTPUT:
- Subset of tokens matching specified tense criteria

In [1]:
# standard library imports 
import ast
# third-party imports
import pandas as pd 

In [2]:
# schema of the 10-character positional morphology encoding
# (cf. index notation) in the "Morphology" column
morphology = {
    "person": { # 0 
        '1': 'first person',
        '2': 'second person',
        '3': 'third person',
        'x': 'uncertain person'
    },
    "number": { # 1
        's': 'singular',
        'd': 'dual',
        'p': 'plural',
        'x': 'uncertain number'
    },
    "tense": { # 2
        'p': 'present',
        'i': 'imperfect',
        'r': 'perfect',
        's': 'resultative',
        'a': 'aorist',
        'u': 'past',
        'l': 'pluperfect',
        'f': 'future',
        't': 'future perfect',
        'x': 'uncertain tense'
    },
    "mood": { # 3 
        'i': 'indicative',
        's': 'subjunctive',
        'm': 'imperative',
        'o': 'optative',
        'n': 'infinitive',
        'p': 'participle',
        'd': 'gerund',
        'g': 'gerundive',
        'u': 'supine',
        'x': 'uncertain mood',
        'y': 'finiteness unspecified',
        'e': 'indicative or subjunctive',
        'f': 'indicative or imperative',
        'h': 'subjunctive or imperative',
        't': 'finite'
    },
    "voice": { # 4
        'a': 'active',
        'm': 'middle',
        'p': 'passive',
        'e': 'middle or passive',
        'x': 'unspecified'
    },
    "gender": { # 5
        'm': 'masculine',
        'f': 'feminine',
        'n': 'neuter',
        'p': 'masculine or feminine',
        'o': 'masculine or neuter',
        'r': 'feminine or neuter',
        'q': 'masculine, feminine or neuter',
        'x': 'uncertain gender'
    },
    "case": { # 6 
        'n': 'nominative',
        'a': 'accusative',
        'o': 'oblique',
        'g': 'genitive',
        'c': 'genitive or dative',
        'e': 'accusative or dative',
        'd': 'dative',
        'b': 'ablative',
        'i': 'instrumental',
        'l': 'locative',
        'v': 'vocative',
        'x': 'uncertain case',
        'z': 'no case'
    },
    "degree": { # 7
        'p': 'positive',
        'c': 'comparative',
        's': 'superlative',
        'x': 'uncertain degree',
        'z': 'no degree'
    },
    "strength": { # 8 
        'w': 'weak',
        's': 'strong',
        't': 'weak or strong'
    },
    "inflection": { # 9 
        'n': 'non-inflecting',
        'i': 'inflecting'
    }
}

In [3]:
# read input file 
df = pd.read_csv("OUTPUTS/dataframe_02_6.csv", 
                 dtype={
                     "Russian Translation": "string",
                     "English Translation": "string", 
                     "Sentence ID": "Int64"
                 }
)

def drop_unnamed_columns(df):
    """
    Drop columns whose names start with 'Unnamed', typically created
    when saving/loading DataFrames with an index in CSV format.
    """
    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

df = drop_unnamed_columns(df)

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_68466/1640784324.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("OUTPUTS/dataframe_02_6.csv",


In [4]:
# create sub dataframe consisting of only verbs

df_verbs = df[
    # keep only verbs 
    (df['POS'] == 'V-') 
    # remove auxiliary verbs 
    # as main verbs already cointain info on its corresponding aux verbs
    & (df["Is_Compound_Aux"]==False)]
# amount of verb forms 
verb_count = len(df_verbs)
verb_count

41525

In [5]:
# file type of the columns "col" is simple string 
# to access elements, convert the columns in list "col" to an actual list 
cols = [
    "Compound_Aux_Forms",
    "Compound_Aux_IDs",
    "Compound_Aux_Lemmas",
    "Compound_Aux_Morphology"
]

for col in cols:
    df_verbs.loc[:, col] = df_verbs[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [6]:
# assert that all columns starting with "Compound..." contain values of type "list" 
for col in df_verbs:
    if col.startswith("Compound"):
        col_type = df_verbs[col].apply(type).value_counts()
        #print(col_type)
        assert (df_verbs[col].apply(lambda x: isinstance(x, list))).all()

In [7]:
# make sure all Compound_Aux verbs have been removed
assert not df_verbs['Is_Compound_Aux'].any()

### CREATE QUERY

In [12]:
FEATURE_INDEX = {
    "person": 0,
    "number": 1,
    "tense": 2,
    "mood": 3,
    "voice": 4,
    "gender": 5,
    "case": 6,
    "degree": 7,
    "strength": 8,
    "inflection": 9
}

def has_feature(morph, feature, value):
    idx = FEATURE_INDEX[feature]
    return isinstance(morph, str) and len(morph) > idx and morph[idx] == value


def match_morph(morph, constraints):
    return all(has_feature(morph, f, v) for f, v in constraints.items())


def find_forms(df, AUX_LEMMAS=None, AUX_FORMS=None, AUX_MORPH=None):

    def match(row):
        x_lem   = row["Compound_Aux_Lemmas"]
        x_forms = row["Compound_Aux_Forms"]
        x_morph = row["Compound_Aux_Morphology"]

        # --- basic sanity ---
        if not (isinstance(x_lem, list) and isinstance(x_morph, list)):
            return False

        # --- length consistency ---
        n = len(x_lem)
        if AUX_LEMMAS and len(AUX_LEMMAS) != n:
            return False
        if AUX_FORMS and len(AUX_FORMS) != n:
            return False
        if AUX_MORPH and len(AUX_MORPH) != n:
            return False

        # --- lemmas (positionsgebunden) ---
        if AUX_LEMMAS:
            if x_lem != AUX_LEMMAS:
                return False

        # --- forms (positionsgebunden) ---
        if AUX_FORMS:
            if not (isinstance(x_forms, list) and x_forms == AUX_FORMS):
                return False

        # --- morphology (positionsgebunden) ---
        if AUX_MORPH:
            for m, constraints in zip(x_morph, AUX_MORPH):
                if not match_morph(m, constraints):
                    return False

        return True

    return df[df.apply(match, axis=1)]


In [13]:
find_forms(
    df_verbs,
    AUX_LEMMAS=["быти", "быти"], # aux1, aux2 
    AUX_MORPH=[
        {"tense": "p", "person": "2"}, # aux1 
        {"tense": "s"} # aux2 
    ]
)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
227498,suz-lav,6685,orv,212117,2295936,шелъ,ити,ити,False,False,...,False,NaN,Moscow,ѥси былъ шелъ,True,False,"[ѥси, былъ]","[2295937, 2295938]","[быти, быти]","[2spia----i, -sspamn-si]"


In [14]:
find_forms(
    df_verbs,
    AUX_LEMMAS=["быти"], # aux1 
    AUX_MORPH=[
        {"tense": "p", "person": "2"}, # aux1 
    ]
)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
487,birchbark,109,orv,210149,2287509,кѹпилъ,купити,купити,False,False,...,False,NaN,Novgorod,еси кѹпилъ робѹ,True,False,[еси],[2287510],[быти],[2spia----i]
537,birchbark,109,orv,210162,2287557,възалъ,възяти,в_зяти,False,False,...,True,не,Novgorod,тꙑ еси не възалъ кѹнъ (техъ),True,False,[еси],[2287555],[быти],[2spia----i]
569,birchbark,142,orv,210170,2287587,дъкънчалъ,доконьчати,докон_чати,False,False,...,False,NaN,Novgorod,ѥси дъкънчалъ,True,False,[ѥси],[2287586],[быти],[2spia----i]
728,birchbark,497,orv,210191,2287737,поихали,поѣхати,поехати,False,False,...,False,NaN,Novgorod,есте поихали,True,False,[есте],[2287736],[быти],[2ppia----i]
815,birchbark,105,orv,210259,2288142,казале,казати,казати,False,False,...,False,NaN,Novgorod,ѥси казале,True,False,[ѥси],[2288141],[быти],[2spia----i]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224294,suz-lav,6683,orv,211377,2288669,подъвели,подъвести,под_вести,False,False,...,False,NaN,Moscow,вы есте подъвели ст҃ослава,True,False,[есте],[2288668],[быти],[2ppia----i]
224622,suz-lav,6683,orv,210427,2288991,всприꙗлъ,въсприяти,в_сприяти,False,False,...,False,NaN,Moscow,ѥси всприꙗлъ вѣнець (побѣдныи),True,False,[ѥси],[2288992],[быти],[2spia----i]
226659,suz-lav,6684,orv,210943,2291878,ѿкрылъ,отъкрыти,от_крыти,False,False,...,False,NaN,Moscow,ѥси ѿкрылъ,True,False,[ѥси],[2291879],[быти],[2spia----i]
227493,suz-lav,6685,orv,211533,2295931,ѹдарилъ,ударити,ударити,False,False,...,False,NaN,Moscow,ѥси ѹдарилъ новъгородъ,True,False,[ѥси],[2295932],[быти],[2spia----i]


In [15]:
find_forms(
    df_verbs,
    AUX_LEMMAS=["быти", "быти"], # aux1, aux2 
)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
3266,pskov,Untitled,orv,207300,1794635,изволили,изволити,изволити,False,False,...,False,NaN,Moscow,"вы бы есте изволили воли (две, мои)",True,False,"[бы, есте]","[1794630, 1794631]","[быти, быти]","[3saia----i, 2ppia----i]"
3280,pskov,Untitled,orv,139964,1794648,сняли,съняти,с_няти,False,False,...,False,NaN,Moscow,бы есте сняли колокол (вечной),True,False,"[бы, есте]","[1794646, 1794647]","[быти, быти]","[3saia----i, 2ppia----i]"
29114,lav,6494: Vladimir consults with representatives o...,orv,126590,1724781,правили,правити,правити,False,False,...,True,не,Nizhny_Novgorod,суть б҃ы не правили вѣры,True,False,"[суть, б҃ы]","[1724778, 1724780]","[быти, быти]","[3ppia----i, 3saia----i]"
52228,lav,Untitled,orv,156165,1747031,пришли,приити,приити,False,False,...,False,NaN,Nizhny_Novgorod,бы ѥсте пришли,True,False,"[бы, ѥсте]","[1747029, 1747030]","[быти, быти]","[3saia----i, 2ppia----i]"
71817,smol-pol-lit,Charter of Prince Jurij Svjatoslavich of Smole...,orv,199594,2227895,повоєвали,повоевати,повоевати,False,False,...,False,NaN,Smolensk,были єсмы повоєвали што,True,False,"[были, єсмы]","[2227893, 2227894]","[быти, быти]","[-pspamn-si, 1ppia----i]"
93410,avv,8,orv,185551,2137814,крстили,крьстити,кр_стити,False,False,...,False,NaN,Moscow,бы есте крстили,True,False,"[бы, есте]","[2137812, 2137813]","[быти, быти]","[3saia----i, 2ppia----i]"
125548,drac,The Tale of Dracula,orv,200267,2233879,повѣдалъ,повѣдати,поведати,False,False,...,True,не,NorthernRussia,бы еси не повѣдалъ ȥлато,True,False,"[бы, еси]","[2233875, 2233877]","[быти, быти]","[3saia----i, 2spia----i]"
125999,drac,The Tale of Dracula,orv,200543,2234324,ѿвѣщал,отъвѣщати,от_вещати,False,False,...,True,не,NorthernRussia,бы еси не ѿвѣщал,True,False,"[бы, еси]","[2234319, 2234321]","[быти, быти]","[2saia----i, 2spia----i]"
126003,drac,The Tale of Dracula,orv,200543,2234327,был,быти,быти,False,False,...,False,NaN,NorthernRussia,бы еси был на,True,False,"[бы, еси]","[2234326, 2234328]","[быти, быти]","[2saia----i, 2spia----i]"
164772,domo,29,orv,161482,2022645,знала,знати,знати,False,False,...,False,NaN,Moscow,жена (сама) бы бы знала,True,False,"[бы, бы]","[2022630, 2022644]","[быти, быти]","[3saia----i, 3saia----i]"


In [17]:
find_forms(
    df_verbs,
    AUX_LEMMAS=["быти", "быти"], # aux1, aux2 
    AUX_MORPH=[
        {"tense":"a"},
        {"tense":"a"}
    ]
)

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,V_yva,V_nuti,...,Negation,Negation_Marker,place,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
164772,domo,29,orv,161482,2022645,знала,знати,знати,False,False,...,False,NaN,Moscow,жена (сама) бы бы знала,True,False,"[бы, бы]","[2022630, 2022644]","[быти, быти]","[3saia----i, 3saia----i]"
165559,domo,32,orv,162100,2026150,была,быти,быти,False,False,...,False,NaN,Moscow,"порѧднѧ (всѧкаѧ, своѧ) бы бы была",True,False,"[бы, бы]","[2026141, 2026149]","[быти, быти]","[3saia----i, 3saia----i]"
167784,domo,38,orv,162301,2027900,было,быти,быти,False,False,...,False,NaN,Moscow,сѹды (всѧкїе) пораднѧ (всѧкаѧ) и бь бы было и ...,True,False,"[бь, бы]","[2027890, 2027899]","[быти, быти]","[3saia----i, 3saia----i]"
170898,domo,45,orv,162621,2031248,былъ,быти,быти,False,False,...,False,NaN,Moscow,"дворъ бы бы былъ или гороженъ, тыненъ",True,False,"[бы, бы]","[2031247, 2031253]","[быти, быти]","[3saia----i, 3saia----i]"
171863,domo,48,orv,163478,2036282,была,быти,быти,False,False,...,False,NaN,Moscow,порѧднѧ (всѧ) бы бы была,True,False,"[бы, бы]","[2036274, 2036283]","[быти, быти]","[3saia----i, 3saia----i]"
172393,domo,49,orv,163536,2036793,оустроили,устроити,устроити,False,False,...,False,NaN,Moscow,ключники (сами) повары хлѣбники стрѧпчие (всѧк...,True,False,"[бы, бы]","[2036779, 2036788]","[быти, быти]","[3saia----i, 3saia----i]"
173470,domo,52,orv,163764,2038234,былъ,быти,быти,False,False,...,False,NaN,Moscow,"запасъ (всѧкои) жита (всѧкое, не згноено) и бы...",True,False,"[бы, бы]","[2038231, 2038235]","[быти, быти]","[3saia----i, 3saia----i]"
